# Notebook 7: CUPED - Variance Reduction for A/B Testing

**Controlled-experiment Using Pre-Experiment Data**

## Overview

CUPED is a powerful statistical technique that uses historical customer data to reduce the variance of treatment effect estimates. In simpler terms: if we already know something about customers before the experiment (like their historical spending), we can use that information to get more precise estimates of how the email campaign affected them. Less noise means we can detect smaller effects and need fewer customers in our tests.

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

# Optional plotly for interactive viz
try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb07', exist_ok=True)


In [ ]:
# Load the cleaned Hillstrom dataset
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nTreatment group counts:")
print(df['segment'].value_counts())
print(f"\nBasic statistics:")
print(df[['visit', 'conversion', 'spend', 'history', 'recency']].describe())

## Concept: Why Variance Reduction Matters

When we run an A/B test, we compare the average outcome between groups. But there's always some **noise** (randomness) in customer behavior. Some customers naturally spend more or less based on their history, not because of the email.

**The Problem:** If we just look at raw spending, this natural variation (noise) makes it hard to see the true effect of the email.

**The Solution:** CUPED says: "We already know how much each customer spent historically. Let's adjust for that." By removing the predictable part of customer behavior (based on history), we can see the email's effect more clearly.

**Why This Matters:**
- **Better precision:** Narrower confidence intervals
- **Faster detection:** Can spot real effects with fewer users
- **Cost savings:** Need fewer experiment participants for the same statistical power

### The Math (in Plain Terms)

Instead of comparing raw spend Y, we compare an **adjusted** version:

```
Y_adjusted = Y - θ × (History - Average_History)
```

Where:
- Y = actual customer spending in experiment
- History = customer's historical spending
- θ = a number we calculate that tells us how much historical spending predicts experimental spending
- Average_History = mean historical spending across all customers

The θ is calculated as:
```
θ = Correlation_Strength / How_Much_History_Varies
```

A stronger correlation and less variation in history means a bigger adjustment and more variance reduction.

## The CUPED Method

CUPED (Controlled-experiment Using Pre-Experiment Data) has these key steps:

1. **Choose a covariate** (X): a pre-experiment variable we know predicts the outcome (e.g., historical spending)
2. **Estimate theta (θ)**: quantify how predictive X is for the outcome Y
3. **Adjust the outcome**: Create Y_cuped = Y - θ × (X - E[X])
4. **Run your test** on the adjusted outcome instead of the raw outcome
5. **Calculate variance reduction**: Compare var(Y) vs var(Y_cuped) to see the improvement

### Why It Works

The adjusted outcome has the same treatment effect (because we subtract the same thing from treatment AND control groups), but lower variance (because we removed predictable noise).

In [ ]:
# Step 1: Prepare data for CUPED on spend outcome
# We'll use 'history' (historical spending) as our pre-experiment covariate

# Separate treatment and control (use Mens Email vs No Email)
men_email = df[df['segment'] == 'Mens E-Mail'].copy()
control = df[df['segment'] == 'No E-Mail'].copy()

# Combine for full analysis
treatment_data = pd.concat([men_email, control]).reset_index(drop=True)

print(f"Mens Email sample size: {len(men_email)}")
print(f"Control sample size: {len(control)}")

# Calculate theta: Cov(spend, history) / Var(history)
# We use the control group to estimate the relationship
X_control = control['history'].values
Y_control = control['spend'].values

# Calculate covariance and variance
covariance_xy = np.cov(X_control, Y_control)[0, 1]
variance_x = np.var(X_control, ddof=1)
theta = covariance_xy / variance_x

print(f"\nCovariance(spend, history): {covariance_xy:.4f}")
print(f"Variance(history): {variance_x:.4f}")
print(f"Theta: {theta:.6f}")
print(f"\nInterpretation: For every dollar of history difference,")
print(f"we expect ~${theta:.4f} difference in experimental spend")

In [ ]:
# Step 2: Create CUPED-adjusted spend metric
mean_history = treatment_data['history'].mean()

# Calculate adjusted spend for all observations
treatment_data['spend_cuped'] = (treatment_data['spend'] - 
                                  theta * (treatment_data['history'] - mean_history))

# Calculate variances
var_raw_spend = treatment_data.groupby('segment')['spend'].var()
var_cuped_spend = treatment_data.groupby('segment')['spend_cuped'].var()

print("Variance Comparison for Spend:")
print("\nRaw Spend Variance:")
print(var_raw_spend)
print("\nCUPED-Adjusted Spend Variance:")
print(var_cuped_spend)

# Calculate variance reduction percentage
var_reduction = (1 - var_cuped_spend / var_raw_spend) * 100
print("\nVariance Reduction (%):")
print(var_reduction)

# The treatment effect should be the same
men_email_idx = treatment_data['segment'] == 'Mens E-Mail'
control_idx = treatment_data['segment'] == 'No E-Mail'

effect_raw = treatment_data[men_email_idx]['spend'].mean() - treatment_data[control_idx]['spend'].mean()
effect_cuped = treatment_data[men_email_idx]['spend_cuped'].mean() - treatment_data[control_idx]['spend_cuped'].mean()

print(f"\nTreatment Effect on Raw Spend: ${effect_raw:.2f}")
print(f"Treatment Effect on CUPED Spend: ${effect_cuped:.2f}")
print(f"(Effect is the same, but variance is lower)")

In [ ]:
# Step 3: Run t-test on CUPED-adjusted spend
from scipy.stats import ttest_ind, norm

treatment_spend_cuped = treatment_data[men_email_idx]['spend_cuped']
control_spend_cuped = treatment_data[control_idx]['spend_cuped']

# Two-sample t-test
t_stat, p_value = ttest_ind(treatment_spend_cuped, control_spend_cuped)

# Calculate confidence intervals
n_treat = len(treatment_spend_cuped)
n_control = len(control_spend_cuped)
mean_treat = treatment_spend_cuped.mean()
mean_control = control_spend_cuped.mean()
std_treat = treatment_spend_cuped.std(ddof=1)
std_control = control_spend_cuped.std(ddof=1)
se_diff = np.sqrt(std_treat**2/n_treat + std_control**2/n_control)
df_welch = (std_treat**2/n_treat + std_control**2/n_control)**2 / ((std_treat**2/n_treat)**2/(n_treat-1) + (std_control**2/n_control)**2/(n_control-1))
t_crit = stats.t.ppf(0.975, df_welch)
ci_lower = effect_cuped - t_crit * se_diff
ci_upper = effect_cuped + t_crit * se_diff

print("\n=== T-Test on CUPED-Adjusted Spend ===")
print(f"Mens Email Mean (CUPED): ${mean_treat:.2f}")
print(f"Control Mean (CUPED): ${mean_control:.2f}")
print(f"Effect Size: ${effect_cuped:.2f}")
print(f"95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Significant at α=0.05? {p_value < 0.05}")

In [ ]:
# Step 4: Visualize the variance reduction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw spend by group
axes[0].hist(treatment_data[men_email_idx]['spend'], bins=50, alpha=0.6, label='Mens Email', edgecolor='black')
axes[0].hist(treatment_data[control_idx]['spend'], bins=50, alpha=0.6, label='Control', edgecolor='black')
axes[0].set_xlabel('Spend ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Raw Spend Distribution')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# CUPED-adjusted spend by group
axes[1].hist(treatment_data[men_email_idx]['spend_cuped'], bins=50, alpha=0.6, label='Mens Email', edgecolor='black')
axes[1].hist(treatment_data[control_idx]['spend_cuped'], bins=50, alpha=0.6, label='Control', edgecolor='black')
axes[1].set_xlabel('CUPED-Adjusted Spend ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('CUPED-Adjusted Spend Distribution')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb07/nb07_cuped_variance_reduction.png', dpi=150, bbox_inches='tight')
plt.show()

print("Visualization shows how CUPED creates tighter distributions around zero,")
print("making treatment differences more apparent.")

In [ ]:
# Step 5: Apply CUPED to binary outcome (visit) using recency
# For binary outcomes, we still use the same approach

X_control_recency = control['recency'].values
Y_control_visit = control['visit'].astype(float).values

covariance_visit = np.cov(X_control_recency, Y_control_visit)[0, 1]
variance_recency = np.var(X_control_recency, ddof=1)
theta_visit = covariance_visit / variance_recency

print("\n=== CUPED on Visit (Binary Outcome) ===")
print(f"Using 'recency' as covariate")
print(f"Theta: {theta_visit:.6f}")

# Create adjusted visit metric
mean_recency = treatment_data['recency'].mean()
treatment_data['visit_cuped'] = (treatment_data['visit'].astype(float) - 
                                   theta_visit * (treatment_data['recency'] - mean_recency))

# Compare variance
var_raw_visit = treatment_data.groupby('segment')['visit'].var()
var_cuped_visit = treatment_data.groupby('segment')['visit_cuped'].var()

var_reduction_visit = (1 - var_cuped_visit / var_raw_visit) * 100

print("\nVariance Reduction for Visit:")
print(f"Raw Visit Variance: {var_raw_visit.mean():.6f}")
print(f"CUPED Visit Variance: {var_cuped_visit.mean():.6f}")
print(f"Variance Reduction: {var_reduction_visit.mean():.2f}%")

# Test on adjusted visit
treatment_visit_cuped = treatment_data[men_email_idx]['visit_cuped']
control_visit_cuped = treatment_data[control_idx]['visit_cuped']

t_stat_visit, p_value_visit = ttest_ind(treatment_visit_cuped, control_visit_cuped)
effect_cuped_visit = treatment_visit_cuped.mean() - control_visit_cuped.mean()

print(f"\nEffect on CUPED Visit: {effect_cuped_visit:.4f}")
print(f"P-value: {p_value_visit:.4f}")
print(f"Significant? {p_value_visit < 0.05}")

## Multi-Covariate CUPED Using Regression

Instead of using just one covariate, we can use multiple pre-experiment variables in a linear regression model. This is more powerful when multiple factors predict the outcome.

The approach:
1. Train a linear regression on the control group: spend ~ history + recency + mens + womens + newbie
2. Predict spend for all subjects using this model
3. Create adjusted metric: Y_cuped = Y - (Y_predicted - Mean_Y_predicted)
4. Run analysis on adjusted metric

This captures more variance reduction because multiple variables together predict outcomes better than any single variable.

In [ ]:
# Step 6: Multi-covariate CUPED using regression
from sklearn.preprocessing import StandardScaler

# Prepare features for regression
features = ['history', 'recency', 'mens', 'womens', 'newbie']

# Ensure features exist and are numeric
for feat in features:
    if feat not in treatment_data.columns:
        print(f"Warning: {feat} not in dataframe")
    
X_features = treatment_data[features].copy()
y_spend = treatment_data['spend'].copy()

# Standardize features for better interpretation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Train regression model on control group to estimate relationship
control_mask = treatment_data['segment'] == 'No E-Mail'
X_control_reg = X_scaled[control_mask]
y_control_spend = y_spend[control_mask]

model = LinearRegression()
model.fit(X_control_reg, y_control_spend)

# Predict for all subjects
y_pred_all = model.predict(X_scaled)
y_pred_mean = y_pred_all.mean()

# Create multi-covariate adjusted spend
treatment_data['spend_cuped_multi'] = y_spend - (y_pred_all - y_pred_mean)

# Compare variance reduction
var_raw = treatment_data['spend'].var()
var_cuped_single = treatment_data['spend_cuped'].var()
var_cuped_multi = treatment_data['spend_cuped_multi'].var()

print("=== Variance Reduction Comparison ===")
print(f"Raw Spend Variance: {var_raw:.2f}")
print(f"Single Covariate CUPED Variance: {var_cuped_single:.2f}")
print(f"Multi-Covariate CUPED Variance: {var_cuped_multi:.2f}")
print(f"\nReduction (Single): {(1 - var_cuped_single/var_raw)*100:.2f}%")
print(f"Reduction (Multi): {(1 - var_cuped_multi/var_raw)*100:.2f}%")

# Test effect with multi-covariate CUPED
treatment_spend_cuped_multi = treatment_data[men_email_idx]['spend_cuped_multi']
control_spend_cuped_multi = treatment_data[control_idx]['spend_cuped_multi']

t_stat_multi, p_value_multi = ttest_ind(treatment_spend_cuped_multi, control_spend_cuped_multi)
effect_cuped_multi = treatment_spend_cuped_multi.mean() - control_spend_cuped_multi.mean()

print(f"\nEffect (Multi-Covariate CUPED): ${effect_cuped_multi:.2f}")
print(f"P-value: {p_value_multi:.4f}")

print(f"\nRegression Model R²: {model.score(X_control_reg, y_control_spend):.4f}")
print(f"Feature Coefficients (standardized):")
for feat, coef in zip(features, model.coef_):
    print(f"  {feat}: {coef:.4f}")

## Variance Reduction Quantification

By reducing variance, CUPED allows us to detect smaller effects or use fewer sample sizes. Here's how:

**Statistical Power:** The ability to detect a true effect if it exists
- Larger variance = lower power (harder to detect effects)
- Smaller variance = higher power (easier to detect effects)

**Sample Size Calculation:** 
For a fixed power level (e.g., 80%), sample size is inversely related to variance reduction.

```
n_needed_after_cuped ≈ n_needed_before × (variance_after / variance_before)
```

**Confidence Interval Width:**
CI width is proportional to sqrt(variance). Smaller variance = tighter CIs.

In [ ]:
# Step 7: Quantify how much sample size is reduced
from scipy.stats import norm

# Assume we want 80% power to detect a $20 effect with alpha=0.05 (two-tailed)
target_effect = 20
alpha = 0.05
power = 0.80

# For t-test, approximate sample size with:
# n = 2 * sigma^2 * (z_alpha + z_beta)^2 / effect^2

z_alpha = norm.ppf(1 - alpha/2)  # ~1.96
z_beta = norm.ppf(power)  # ~0.84

# Raw spend variance
var_spend_raw = treatment_data['spend'].var()
n_raw = 2 * var_spend_raw * (z_alpha + z_beta)**2 / (target_effect**2)

# CUPED (single) variance
n_cuped_single = 2 * var_cuped_single * (z_alpha + z_beta)**2 / (target_effect**2)

# CUPED (multi) variance
n_cuped_multi = 2 * var_cuped_multi * (z_alpha + z_beta)**2 / (target_effect**2)

print("=== Sample Size to Detect $20 Effect (80% Power) ===")
print(f"Raw Spend Approach: {n_raw:.0f} customers per group ({n_raw*2:.0f} total)")
print(f"Single-Cov CUPED: {n_cuped_single:.0f} customers per group ({n_cuped_single*2:.0f} total)")
print(f"Multi-Cov CUPED: {n_cuped_multi:.0f} customers per group ({n_cuped_multi*2:.0f} total)")

savings_single = (1 - n_cuped_single/n_raw) * 100
savings_multi = (1 - n_cuped_multi/n_raw) * 100

print(f"\nSample Size Reduction:")
print(f"Single-Cov CUPED: {savings_single:.1f}% fewer customers needed")
print(f"Multi-Cov CUPED: {savings_multi:.1f}% fewer customers needed")

# Visualize CI width reduction
ci_width_raw = 1.96 * 2 * np.sqrt(var_spend_raw / (len(men_email)/2))
ci_width_cuped = 1.96 * 2 * np.sqrt(var_cuped_single / (len(men_email)/2))
ci_width_cuped_multi = 1.96 * 2 * np.sqrt(var_cuped_multi / (len(men_email)/2))

print(f"\n95% CI Width (assuming current sample size):")
print(f"Raw Spend: ±${ci_width_raw/2:.2f}")
print(f"CUPED (Single): ±${ci_width_cuped/2:.2f}")
print(f"CUPED (Multi): ±${ci_width_cuped_multi/2:.2f}")

In [ ]:
# Visualize confidence interval width improvements
methods = ['Raw Spend', 'CUPED
(Single Cov)', 'CUPED
(Multi Cov)']
ci_widths = [ci_width_raw, ci_width_cuped, ci_width_cuped_multi]
reductions = [0, savings_single, savings_multi]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# CI width comparison
colors = ['#d62728', '#ff7f0e', '#2ca02c']
bars = ax1.bar(methods, ci_widths, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('95% CI Width (±$)', fontsize=12)
ax1.set_title('Confidence Interval Width by Method', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, width in zip(bars, ci_widths):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'±${height/2:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Sample size reduction
bars2 = ax2.bar(methods, reductions, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Sample Size Reduction (%)', fontsize=12)
ax2.set_title('Sample Size Savings vs Raw Approach', fontsize=14, fontweight='bold')
ax2.set_ylim(0, max(reductions) * 1.2)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar, reduction in zip(bars2, reductions):
    if reduction > 0:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{reduction:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/outputs/nb07/nb07_cuped_ci_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## Comparison: Naive vs CUPED Results

Here we create a side-by-side comparison of all our methods to see how CUPED improves our statistical power.

In [ ]:
# Step 8: Comprehensive comparison table
comparison_results = []

# Raw spend test
t_stat_raw, p_value_raw = ttest_ind(
    treatment_data[men_email_idx]['spend'],
    treatment_data[control_idx]['spend']
)
effect_raw_spend = (treatment_data[men_email_idx]['spend'].mean() - 
                    treatment_data[control_idx]['spend'].mean())
se_raw = np.sqrt(var_spend_raw * (2/len(men_email)))
ci_raw_spend = (effect_raw_spend - 1.96*se_raw, effect_raw_spend + 1.96*se_raw)

comparison_results.append({
    'Method': 'Raw Spend',
    'Effect': f'${effect_raw_spend:.2f}',
    'SE': f'${se_raw:.2f}',
    'CI_Lower': f'${ci_raw_spend[0]:.2f}',
    'CI_Upper': f'${ci_raw_spend[1]:.2f}',
    'P-Value': f'{p_value_raw:.4f}',
    'Significant': 'Yes' if p_value_raw < 0.05 else 'No',
    'Variance': f'{var_spend_raw:.2f}'
})

# CUPED (single covariate)
se_cuped = np.sqrt(var_cuped_single * (2/len(men_email)))
ci_cuped_spend = (effect_cuped - 1.96*se_cuped, effect_cuped + 1.96*se_cuped)

comparison_results.append({
    'Method': 'CUPED (History)',
    'Effect': f'${effect_cuped:.2f}',
    'SE': f'${se_cuped:.2f}',
    'CI_Lower': f'${ci_cuped_spend[0]:.2f}',
    'CI_Upper': f'${ci_cuped_spend[1]:.2f}',
    'P-Value': f'{p_value:.4f}',
    'Significant': 'Yes' if p_value < 0.05 else 'No',
    'Variance': f'{var_cuped_single:.2f}'
})

# CUPED (multi-covariate)
se_cuped_multi = np.sqrt(var_cuped_multi * (2/len(men_email)))
ci_cuped_multi_spend = (effect_cuped_multi - 1.96*se_cuped_multi, effect_cuped_multi + 1.96*se_cuped_multi)

comparison_results.append({
    'Method': 'CUPED (Multi)',
    'Effect': f'${effect_cuped_multi:.2f}',
    'SE': f'${se_cuped_multi:.2f}',
    'CI_Lower': f'${ci_cuped_multi_spend[0]:.2f}',
    'CI_Upper': f'${ci_cuped_multi_spend[1]:.2f}',
    'P-Value': f'{p_value_multi:.4f}',
    'Significant': 'Yes' if p_value_multi < 0.05 else 'No',
    'Variance': f'{var_cuped_multi:.2f}'
})

comparison_df = pd.DataFrame(comparison_results)
print("\n=== COMPREHENSIVE RESULTS COMPARISON ===\n")
print(comparison_df.to_string(index=False))

# Save results
comparison_df.to_csv('../data/outputs/nb07/nb07_cuped_results.csv', index=False)
print("\nResults saved to: ../data/outputs/nb07/nb07_cuped_results.csv")

## Key Takeaways

1. **Variance Reduction Works**: CUPED reduces variance by leveraging pre-experiment data
2. **Same Effect Estimate**: The treatment effect doesn't change, but precision improves
3. **Practical Benefits**: 
   - Narrower confidence intervals
   - Fewer customers needed for same power
   - Ability to detect smaller effects
4. **Multi-Covariate is Best**: Using multiple predictors (history, recency, etc.) provides the most variance reduction
5. **When to Use CUPED**: 
   - When you have good pre-experiment data
   - When outcome variance is high
   - When you want to detect small effects efficiently

## Next Steps

- Consider CUPED for all future experiments
- Choose covariates based on their predictive power
- Validate results with out-of-sample data
- Document baseline metrics for future experiments